# Feature Creation Notebook

Authors: Harry Zhang, Thomas Saleh
Packages authored by: Idil Yaktubay, Amanbir Singh, Harry Zhang, Thomas Saleh 

CODE PURPOSE: Create feature tables for total dissolved solids, turbidity, total organic carbon, and discharge at Pinawa and Whitemouth AquaHive stations, for the purpose of parameter forecasting

## Set-up
Import packages

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
import pandas as pd
import os
import sys
sys.path.append('..')
def feature_checks(feature_df):
    
    print('columns %s' % feature_df.columns)
    print('start time %s' % feature_df.index[1])
    print('end time %s' % feature_df.index[-1])
    print('number of rows %s and columns %f' % feature_df.shape)
    print('Count of non-null values by column')
    
    return (feature_df.notna().sum())
def convert_timestamps(df):
    '''
    Return df with new column containing timestamps converted to Canada/Central,
    taking DST into account
    Parameters
    ----------
    df : DataFrame
            AquaHive Dataset
    Returns
    -------
    df_out: DataFrame
        df with additional converted timestamp column
    '''
    
    df_out = df.copy()
    
    
    df_out['timestamp_ccentral'] = \
        pd.to_datetime(df_out['timestamp']).dt.tz_convert('Canada/Central')
    
    
    return df_out

    
def calc_param_site_features_hrs_ago(df, param: str, site: str,
                                     hours_ago_start: int, hours_ago_end: int):
    '''
    Return datasets containing calculated features for parameter param and 
    site site records in df for hours_ago_start-hours_ago_end hours ago bucket

    Parameters
    ----------
    df : DataFrame
        AquaHive dataset
    param : str
        The parameter to filter for (e.g., 'tn' for total nitrogen)
    site : str
        The site to filter for (e.g., 'pnwa' for Pinawa)
    hours_ago_start: int
        The start of time bucket (e.g., 4 for 4-6h ago time bucket)
    hours_ago_end: int
        The end of time bucket (e.g., 6 for 4-6h ago time bucket)

    Returns
    -------
    final_df_conc, final_df_load: Tuple of two DataFrames
        First dataframe is feature table for nutrient concentration
        Second dataframe is feature table for nutrient load

    '''
    # Make copy of original dataset to avoid modifying it
    data_out = df.copy()
    
    
    # Filter for parameter and site and create datetime type timestamp column 
        # (Canada Central)
    data_out = convert_timestamps(calc_nutrient_load(
            filter_for_param_site(data_out, param, site), param))
    
    
    # Sort timestamps in ascending order
    data_out = data_out.sort_values('timestamp_ccentral', ascending=True)
    
    
    # Calculate number of records in hours-ago range
    hours_offset = (hours_ago_end - hours_ago_start) + 1
    
    
    # Create rolling windows for hours_offset bucket
    rolling_window = data_out[['value', 'load', 'timestamp_ccentral']].rolling(
                     window=str(hours_offset)+'h', 
                     on='timestamp_ccentral',
                     min_periods=1, 
                     closed = 'left')
    
    # Calculate the number of positions to shift depending on hours-ago range
    shift_scalar = hours_ago_start//hours_offset
    shift_num = shift_scalar*hours_offset
    
    
    # Calculate mean, max, min, and max-min range for param concentration
        # Not calculating variance since windows are too small
    mean = rolling_window['value'].mean().shift(shift_num)
    # min_ = rolling_window['value'].min().shift(shift_num)
    # max_ = rolling_window['value'].max().shift(shift_num)
    # max_min = abs(max_ - min_)
    
    
    # Calculate mean, max, min, and max-min range for param load
        # Not calculating variance since windows are too small
    mean_load = rolling_window['load'].mean().shift(shift_num)
    # min_load = rolling_window['load'].min().shift(shift_num)
    # max_load = rolling_window['load'].max().shift(shift_num)
    # max_min_load = abs(max_load - min_load)
    
    
    #  Add new columns with calculated features for concentration and load
    final_df_conc = pd.concat([mean], #min_, max_, max_min],
                              axis=1)
    final_df_load = pd.concat([mean_load], #min_load, max_load, max_min_load],
                              axis=1)
    cols_conc = [
            x+'_'+str(hours_ago_start)+'_'+str(hours_ago_end)
            for x in [param+'_'+site+'_mean']
                    #   param+'_'+site+'_min', 
                    #   param+'_'+site+'_max', 
                    #   param+'_'+site+'_max_min']
                ]
    
    cols_load = [
            x+'_'+str(hours_ago_start)+'_'+str(hours_ago_end)
            for x in [param+'_'+site+'_mean_load']
                    #   param+'_'+site+'_min_load', 
                    #   param+'_'+site+'_max_load', 
                    #   param+'_'+site+'_max_min_load']
                ]
            
    final_df_conc.columns = cols_conc
    final_df_load.columns = cols_load
    
    
    # Add timestamp column
    final_df_conc.loc[:, 'timestamp_ccentral'] = data_out['timestamp_ccentral']
    final_df_load.loc[:, 'timestamp_ccentral'] = data_out['timestamp_ccentral']
    
    
    # return a tuple of two separate dataframes, one concentration one load
    return (final_df_conc, final_df_load)


def calc_discharge_site_features_hrs_ago(df, site: str, hours_ago_start: int,
                                         hours_ago_end: int):
    '''
    Return dataset containing calculated discharge features for site site
    records in df for hours_ago_start-hours_ago_end hours ago bucket

    Parameters
    ----------
    df : DataFrame
        AquaHive Dataset
    site : str
        The site to filter for (e.f., 'pnwa' for Pinawa)
    hours_ago_start : int
        The start of time bucket (e.g., 4 for 4-6h ago time bucket)
    hours_ago_end : int
        The end of time bucket (e.g., 6 for 4-6h ago time bucket)

    Returns
    -------
    final_df: Dataframe
        Dataset containing desired features

    '''
    # Make copy of original dataset to avoid modifying it
    data_out = df.copy()
    
    
    # Filter for site and create converted timestamp column
    data_out = convert_timestamps(data_out[data_out['site'] == site])
    
    
    # Drop duplicate timestamps
    data_out = data_out.drop_duplicates(subset=['timestamp_ccentral'])
    
    
    # Sort timestamps in ascending order
    data_out = data_out.sort_values('timestamp_ccentral', ascending=True)
    
    
    # Calculate number of records in hours-ago range
    hours_offset = (hours_ago_end - hours_ago_start) + 1
    
    
    # Create rolling windows for hours_offset bucket
    rolling_window = data_out[['discharge', 'timestamp_ccentral']].rolling(
                     window=str(hours_offset)+'h', 
                     on='timestamp_ccentral',
                     min_periods=1, 
                     closed='left')
    
    
    # Calculate the number of positions to shift depending on hours-ago range
    shift_scalar = hours_ago_start//hours_offset
    shift_num = shift_scalar*hours_offset
    
    
    # Calculate mean, max, min, and max-min range for discharge
        # Not calculating variance since windows are too small
    mean = rolling_window['discharge'].mean().shift(shift_num)
    # min_ = rolling_window['discharge'].min().shift(shift_num)
    # max_ = rolling_window['discharge'].max().shift(shift_num)
    # max_min = abs(max_ - min_)
    
    
    #  Add new columns with calculated features
    final_df = pd.concat([mean], #min_, max_, max_min],
                          axis=1)
    
    cols = [
            x+'_'+str(hours_ago_start)+'_'+str(hours_ago_end)
            for x in ['dschrg_'+site+'_mean']
                    #   'dschrg_'+site+'_min', 
                    #   'dschrg'+site+'_max', 
                    #   'dschrg'+site+'_max_min']
           ]
    
    final_df.columns = cols
    final_df.loc[:, 'timestamp_ccentral'] = data_out['timestamp_ccentral']
    
    return final_df
    
def calc_nutrient_load(df, param: str):
    '''
    Return df with new column containing calculated nutrient load values

    Parameters
    ----------
    df : DataFrame
        AquaHive Dataset
    param: str
        The parameter that df contains (df must only contain data for a single
                                        parameter for unit consistency)
    Returns
    -------
    df_out : DataFrame
        df with additional nutrient load column

    '''
    df_out = df.copy()
    
    # Unit conversions for parameters not listed in mg/L units
    if param == 'turbidity':
        # Convert turbidity units from NTU to mg/L
        df_out['value_converted'] = df_out['value']/3.
        df_out['load'] = df_out['value_converted']*df_out['discharge']
        df_out.drop(columns=['value_converted'], inplace=True)  
    # if param == 'total_dissolved_solids':
        # Convert total dissolved solids units from ppt to mg/L
        # ... Add some code here and uncomment when we need tds eventually...
    else: 
        # This calculation assumes that parameter units are mg/L
        df_out['load'] = df_out['value']*df_out['discharge']
    
    
    return df_out

def filter_for_param_site(df, param: str, site: str):
    '''
    Filter full AquaHive dataset for parameter param and site site

    Parameters
    ----------
    df : DataFrame
        AquaHive dataset
    param: str
        The parameter to filter for (e.g., 'tn' for total nitrogen)
    site: str
        The AquahHive site to filter for (e.g., 'pnwa' for Pinawa)

    Returns
    -------
    df_out: DataFrame
        Input AquaHive Dataset filtered for param and site
    '''
    df_out = df.copy()
    
    
    df_out = df_out[
                (df_out['parameter'] == param) &
                (df_out['site'] == site)
                ]
    
    
    return df_out


Import raw aquahive data

In [52]:
full_dataset = pd.read_csv("met_ts_t2m_cdir_CR.csv")

## Generate turbidity datasets

Whitemouth

In [72]:
t2m_cr_1_12_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=1, 
                                                        hours_ago_end=12)

t2m_cr_13_48_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=13, 
                                                        hours_ago_end=48)

t2m_cr_49_120_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=49, 
                                                        hours_ago_end=120)

t2m_cr_121_720_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=121, 
                                                        hours_ago_end=720)


Combine turbidity features from different time windows into a single table

In [73]:
# Create final turbidity concentration feature table (wmth)
t2m_cr_final_features_conc = pd.concat([t2m_cr_1_12_features[0].set_index('timestamp_ccentral'),
                                        t2m_cr_13_48_features[0].set_index('timestamp_ccentral'),
                                        t2m_cr_49_120_features[0].set_index('timestamp_ccentral'),
                                        t2m_cr_121_720_features[0].set_index('timestamp_ccentral')
                                        ],
                                        axis=1)
# Create final turbidity load feature table (wmth)
t2m_cr_final_features_load = pd.concat([t2m_cr_1_12_features[1].set_index('timestamp_ccentral'),
                                          t2m_cr_13_48_features[1].set_index('timestamp_ccentral'),
                                          t2m_cr_49_120_features[1].set_index('timestamp_ccentral'),
                                          t2m_cr_121_720_features[1].set_index('timestamp_ccentral')
                                          ],
                                          axis=1)

Reviewing turbidity features dataset

In [74]:
feature_checks(t2m_cr_final_features_conc)
t2m_cr_final_features_conc.head()

columns Index(['t2m_cr_mean_1_12', 't2m_cr_mean_13_48', 't2m_cr_mean_49_120',
       't2m_cr_mean_121_720'],
      dtype='object')
start time 2015-12-31 19:00:00-06:00
end time 2020-12-31 17:00:00-06:00
number of rows 43848 and columns 4.000000
Count of non-null values by column


,t2m_cr_mean_1_12,t2m_cr_mean_13_48,t2m_cr_mean_49_120,t2m_cr_mean_121_720
timestamp_ccentral,,,,
2015-12-31 18:00:00-06:00,NaN,NaN,NaN,NaN
2015-12-31 19:00:00-06:00,265.051697,265.051697,265.051697,265.051697
2015-12-31 20:00:00-06:00,265.126160,265.126160,265.126160,265.126160
2015-12-31 21:00:00-06:00,265.226898,265.226898,265.226898,265.226898
2015-12-31 22:00:00-06:00,265.602371,265.602371,265.602371,265.602371


In [71]:
# Export datasets as csv (uncomment if need to re-export)
t2m_cr_final_features_conc.to_csv("t2m_cr_concentration_features_test.csv", index=True)
t2m_cr_final_features_load.to_csv("t2m_cr_load_features.csv_test", index=True)

Pinawa turbidity features

In [60]:
cdir_cr_1_12_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=1, 
                                                        hours_ago_end=3)

cdir_cr_13_48_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=4, 
                                                        hours_ago_end=6)

cdir_cr_49_120_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=7, 
                                                        hours_ago_end=9)

cdir_cr_121_720_features = calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=10, 
                                                        hours_ago_end=12)


# Create final turbidity concentration feature table (pnwa)
cdir_cr_final_features_conc = pd.concat([cdir_cr_1_12_features[0].set_index('timestamp_ccentral'),
                                        cdir_cr_13_48_features[0].set_index('timestamp_ccentral'),
                                        cdir_cr_49_120_features[0].set_index('timestamp_ccentral'),
                                        cdir_cr_121_720_features[0].set_index('timestamp_ccentral')
                                        ],
                                        axis=1)

# Create final turbidity load feature table (pnwa)
cdir_cr_final_features_load = pd.concat([cdir_cr_1_12_features[1].set_index('timestamp_ccentral'),
                                        cdir_cr_13_48_features[1].set_index('timestamp_ccentral'),
                                        cdir_cr_49_120_features[1].set_index('timestamp_ccentral'),
                                        cdir_cr_121_720_features[1].set_index('timestamp_ccentral')
                                        ],
                                        axis=1)

# Export datasets as csv (uncomment if need to re-export)
cdir_cr_final_features_conc.to_csv("cdir_cr_concentration_features.csv", index=True)
cdir_cr_final_features_load.to_csv("cdir_cr_load_features.csv", index=True)

Reviewing Pinawa turbidity dataset

In [61]:
feature_checks(cdir_cr_final_features_conc)
cdir_cr_final_features_conc.head()
#tur_pnwa_final_features_conc.dropna(axis = 0).head()

columns Index(['cdir_cr_mean_1_3', 'cdir_cr_mean_4_6', 'cdir_cr_mean_7_9',
       'cdir_cr_mean_10_12'],
      dtype='object')
start time 2015-12-31 19:00:00-06:00
end time 2020-12-31 17:00:00-06:00
number of rows 43848 and columns 4.000000
Count of non-null values by column


,cdir_cr_mean_1_3,cdir_cr_mean_4_6,cdir_cr_mean_7_9,cdir_cr_mean_10_12
timestamp_ccentral,,,,
2015-12-31 18:00:00-06:00,NaN,NaN,NaN,NaN
2015-12-31 19:00:00-06:00,0.0,NaN,NaN,NaN
2015-12-31 20:00:00-06:00,0.0,NaN,NaN,NaN
2015-12-31 21:00:00-06:00,0.0,NaN,NaN,NaN
2015-12-31 22:00:00-06:00,0.0,0.0,NaN,NaN


## Generate TOC Datasets
Whitemouth

In [10]:
toc_wmth_1_3_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=1,
                                                        hours_ago_end=3)
 
toc_wmth_4_6_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=4,
                                                        hours_ago_end=6)
 
toc_wmth_7_9_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=7,
                                                        hours_ago_end=9)
 
toc_wmth_10_12_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=10,
                                                        hours_ago_end=12)
 
toc_wmth_13_15_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=13,
                                                        hours_ago_end=15)
 
toc_wmth_16_18_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=16,
                                                        hours_ago_end=18)
 
toc_wmth_19_21_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=19,
                                                        hours_ago_end=21)
 
toc_wmth_22_24_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=22,
                                                        hours_ago_end=24)
 
toc_wmth_25_27_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=25,
                                                        hours_ago_end=27)
 
toc_wmth_28_30_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=28,
                                                        hours_ago_end=30)

toc_wmth_31_33_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=31,
                                                        hours_ago_end=33)
 
toc_wmth_34_36_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=34,
                                                        hours_ago_end=36)
 
toc_wmth_37_39_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=37,
                                                        hours_ago_end=39)
 
toc_wmth_40_42_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=40,
                                                        hours_ago_end=42)
 
toc_wmth_43_45_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=43,
                                                        hours_ago_end=45)
 
toc_wmth_46_48_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='wmth',
                                                        hours_ago_start=46,
                                                        hours_ago_end=48)

# Create final total organic carbon concentration feature table (wmth)
toc_wmth_final_features_conc = pd.concat([toc_wmth_1_3_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_4_6_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_7_9_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_10_12_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_13_15_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_16_18_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_19_21_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_22_24_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_25_27_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_28_30_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_31_33_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_34_36_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_37_39_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_40_42_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_43_45_features[0].set_index('timestamp_ccentral'),
                                          toc_wmth_46_48_features[0].set_index('timestamp_ccentral')
                                          ],
                                          axis=1)

# Create final total organic carbon load feature table (wmth)
toc_wmth_final_features_load = pd.concat([toc_wmth_1_3_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_4_6_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_7_9_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_10_12_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_13_15_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_16_18_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_19_21_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_22_24_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_25_27_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_28_30_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_31_33_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_34_36_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_37_39_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_40_42_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_43_45_features[1].set_index('timestamp_ccentral'),
                                          toc_wmth_46_48_features[1].set_index('timestamp_ccentral')
                                          ],
                                          axis=1)

In [11]:
# Export datasets as csv (uncomment if need to re-export)
toc_wmth_final_features_conc.to_csv(os.path.join(project_conf.INTERIM_DATA_DIR, 'toc_wmth_concentration_features.csv'), index=True)
toc_wmth_final_features_load.to_csv(os.path.join(project_conf.INTERIM_DATA_DIR, 'toc_wmth_load_features.csv'), index=True)

Check dataset

In [12]:
feature_checks(toc_wmth_final_features_conc)
toc_wmth_final_features_conc.head()

columns Index(['toc_wmth_mean_1_3', 'toc_wmth_mean_4_6', 'toc_wmth_mean_7_9',
       'toc_wmth_mean_10_12', 'toc_wmth_mean_13_15', 'toc_wmth_mean_16_18',
       'toc_wmth_mean_19_21', 'toc_wmth_mean_22_24', 'toc_wmth_mean_25_27',
       'toc_wmth_mean_28_30', 'toc_wmth_mean_31_33', 'toc_wmth_mean_34_36',
       'toc_wmth_mean_37_39', 'toc_wmth_mean_40_42', 'toc_wmth_mean_43_45',
       'toc_wmth_mean_46_48'],
      dtype='object')
start time 2022-03-11 22:00:00-06:00
end time 2023-10-29 04:00:00-05:00
number of rows 14314 and columns 16.000000
Count of non-null values by column


,toc_wmth_mean_1_3,toc_wmth_mean_4_6,toc_wmth_mean_7_9,toc_wmth_mean_10_12,toc_wmth_mean_13_15,toc_wmth_mean_16_18,toc_wmth_mean_19_21,toc_wmth_mean_22_24,toc_wmth_mean_25_27,toc_wmth_mean_28_30,toc_wmth_mean_31_33,toc_wmth_mean_34_36,toc_wmth_mean_37_39,toc_wmth_mean_40_42,toc_wmth_mean_43_45,toc_wmth_mean_46_48
timestamp_ccentral,,,,,,,,,,,,,,,,
2022-03-11 21:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-11 22:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-11 23:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-12 00:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-12 01:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Pinawa

In [13]:
toc_pnwa_1_3_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=1,
                                                        hours_ago_end=3)
 
toc_pnwa_4_6_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=4,
                                                        hours_ago_end=6)
 
toc_pnwa_7_9_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=7,
                                                        hours_ago_end=9)
 
toc_pnwa_10_12_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=10,
                                                        hours_ago_end=12)
 
toc_pnwa_13_15_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=13,
                                                        hours_ago_end=15)
 
toc_pnwa_16_18_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=16,
                                                        hours_ago_end=18)
 
toc_pnwa_19_21_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=19,
                                                        hours_ago_end=21)
 
toc_pnwa_22_24_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=22,
                                                        hours_ago_end=24)
 
toc_pnwa_25_27_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=25,
                                                        hours_ago_end=27)
 
toc_pnwa_28_30_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=28,
                                                        hours_ago_end=30)

toc_pnwa_31_33_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=31,
                                                        hours_ago_end=33)
 
toc_pnwa_34_36_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=34,
                                                        hours_ago_end=36)
 
toc_pnwa_37_39_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=37,
                                                        hours_ago_end=39)
 
toc_pnwa_40_42_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=40,
                                                        hours_ago_end=42)
 
toc_pnwa_43_45_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=43,
                                                        hours_ago_end=45)
 
toc_pnwa_46_48_features = calc_param_site_features_hrs_ago(df=full_dataset,
                                                        param='toc',
                                                        site='pnwa',
                                                        hours_ago_start=46,
                                                        hours_ago_end=48)

# Create final total organic carbon concentration feature table (pnwa)
toc_pnwa_final_features_conc = pd.concat([toc_pnwa_1_3_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_4_6_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_7_9_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_10_12_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_13_15_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_16_18_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_19_21_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_22_24_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_25_27_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_28_30_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_31_33_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_34_36_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_37_39_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_40_42_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_43_45_features[0].set_index('timestamp_ccentral'),
                                          toc_pnwa_46_48_features[0].set_index('timestamp_ccentral')
                                          ],
                                          axis=1)

# Create final total organic carbon load feature table (pnwa)
toc_pnwa_final_features_load = pd.concat([toc_pnwa_1_3_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_4_6_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_7_9_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_10_12_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_13_15_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_16_18_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_19_21_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_22_24_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_25_27_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_28_30_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_31_33_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_34_36_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_37_39_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_40_42_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_43_45_features[1].set_index('timestamp_ccentral'),
                                          toc_pnwa_46_48_features[1].set_index('timestamp_ccentral')
                                          ],
                                          axis=1)

Checking dataset

In [14]:
feature_checks(toc_pnwa_final_features_conc)
toc_pnwa_final_features_conc.head()

columns Index(['toc_pnwa_mean_1_3', 'toc_pnwa_mean_4_6', 'toc_pnwa_mean_7_9',
       'toc_pnwa_mean_10_12', 'toc_pnwa_mean_13_15', 'toc_pnwa_mean_16_18',
       'toc_pnwa_mean_19_21', 'toc_pnwa_mean_22_24', 'toc_pnwa_mean_25_27',
       'toc_pnwa_mean_28_30', 'toc_pnwa_mean_31_33', 'toc_pnwa_mean_34_36',
       'toc_pnwa_mean_37_39', 'toc_pnwa_mean_40_42', 'toc_pnwa_mean_43_45',
       'toc_pnwa_mean_46_48'],
      dtype='object')
start time 2020-12-31 09:00:00-06:00
end time 2023-12-31 07:00:00-06:00
number of rows 26408 and columns 16.000000
Count of non-null values by column


,toc_pnwa_mean_1_3,toc_pnwa_mean_4_6,toc_pnwa_mean_7_9,toc_pnwa_mean_10_12,toc_pnwa_mean_13_15,toc_pnwa_mean_16_18,toc_pnwa_mean_19_21,toc_pnwa_mean_22_24,toc_pnwa_mean_25_27,toc_pnwa_mean_28_30,toc_pnwa_mean_31_33,toc_pnwa_mean_34_36,toc_pnwa_mean_37_39,toc_pnwa_mean_40_42,toc_pnwa_mean_43_45,toc_pnwa_mean_46_48
timestamp_ccentral,,,,,,,,,,,,,,,,
2020-12-31 08:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 09:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 10:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 11:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 12:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Export datasets as csv (uncomment if need to re-export)
toc_pnwa_final_features_conc.to_csv(os.path.join(project_conf.INTERIM_DATA_DIR, 'toc_pnwa_concentration_features.csv'), index=True)
toc_pnwa_final_features_load.to_csv(os.path.join(project_conf.INTERIM_DATA_DIR, 'toc_pnwa_load_features.csv'), index=True)


## Discharge features
Whitemouth

In [24]:
dschrg_wmth_1_3_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=1,
                                                        hours_ago_end=3)
 
dschrg_wmth_4_6_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=4,
                                                        hours_ago_end=6)
 
dschrg_wmth_7_9_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=7,
                                                        hours_ago_end=9)
 
dschrg_wmth_10_12_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=10,
                                                        hours_ago_end=12)
 
dschrg_wmth_13_15_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=13,
                                                        hours_ago_end=15)
 
dschrg_wmth_16_18_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=16,
                                                        hours_ago_end=18)
 
dschrg_wmth_19_21_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=19,
                                                        hours_ago_end=21)
 
dschrg_wmth_22_24_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=22,
                                                        hours_ago_end=24)
 
dschrg_wmth_25_27_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=25,
                                                        hours_ago_end=27)
 
dschrg_wmth_28_30_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=28,
                                                        hours_ago_end=30)

dschrg_wmth_31_33_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=31,
                                                        hours_ago_end=33)
 
dschrg_wmth_34_36_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=34,
                                                        hours_ago_end=36)
 
dschrg_wmth_37_39_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=37,
                                                        hours_ago_end=39)
 

# Create final discharge feature table (wmth)
dschrg_wmth_final_features = pd.concat([dschrg_wmth_1_3_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_4_6_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_7_9_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_10_12_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_13_15_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_16_18_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_19_21_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_22_24_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_25_27_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_28_30_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_31_33_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_34_36_features.set_index('timestamp_ccentral'),
                                        dschrg_wmth_37_39_features.set_index('timestamp_ccentral')],
                                          axis=1)


In [25]:
feature_checks(dschrg_wmth_final_features)
dschrg_wmth_final_features.head()
dschrg_wmth_final_features.dropna(axis = 0).head()

columns Index(['dschrg_cr_mean_1_3', 'dschrg_cr_mean_4_6', 'dschrg_cr_mean_7_9',
       'dschrg_cr_mean_10_12', 'dschrg_cr_mean_13_15', 'dschrg_cr_mean_16_18',
       'dschrg_cr_mean_19_21', 'dschrg_cr_mean_22_24', 'dschrg_cr_mean_25_27',
       'dschrg_cr_mean_28_30', 'dschrg_cr_mean_31_33', 'dschrg_cr_mean_34_36',
       'dschrg_cr_mean_37_39'],
      dtype='object')
start time 2016-03-01 19:00:00-06:00
end time 2018-11-01 19:00:00-05:00
number of rows 17640 and columns 13.000000
Count of non-null values by column


,dschrg_cr_mean_1_3,dschrg_cr_mean_4_6,dschrg_cr_mean_7_9,dschrg_cr_mean_10_12,dschrg_cr_mean_13_15,dschrg_cr_mean_16_18,dschrg_cr_mean_19_21,dschrg_cr_mean_22_24,dschrg_cr_mean_25_27,dschrg_cr_mean_28_30,dschrg_cr_mean_31_33,dschrg_cr_mean_34_36,dschrg_cr_mean_37_39
timestamp_ccentral,,,,,,,,,,,,,
2016-03-03 07:00:00-06:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-03-03 08:00:00-06:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-03-03 09:00:00-06:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-03-03 10:00:00-06:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-03-03 11:00:00-06:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# # Export dataset as csv (uncomment if need to re-export)
dschrg_wmth_final_features.to_csv("discharge_wmth_features_2016-2018.csv", index=True)


In [19]:
dschrg_pnwa_1_3_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=1,
                                                        hours_ago_end=3)
 
dschrg_pnwa_4_6_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=4,
                                                        hours_ago_end=6)
 
dschrg_pnwa_7_9_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=7,
                                                        hours_ago_end=9)
 
dschrg_pnwa_10_12_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=10,
                                                        hours_ago_end=12)
 
dschrg_pnwa_13_15_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=13,
                                                        hours_ago_end=15)
 
dschrg_pnwa_16_18_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=16,
                                                        hours_ago_end=18)
 
dschrg_pnwa_19_21_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=19,
                                                        hours_ago_end=21)
 
dschrg_pnwa_22_24_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=22,
                                                        hours_ago_end=24)
 
dschrg_pnwa_25_27_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=25,
                                                        hours_ago_end=27)
 
dschrg_pnwa_28_30_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=28,
                                                        hours_ago_end=30)

dschrg_pnwa_31_33_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=31,
                                                        hours_ago_end=33)
 
dschrg_pnwa_34_36_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=34,
                                                        hours_ago_end=36)
 
dschrg_pnwa_37_39_features = calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='pnwa',
                                                        hours_ago_start=37,
                                                        hours_ago_end=39)
 


dschrg_pnwa_final_features = pd.concat([dschrg_pnwa_1_3_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_4_6_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_7_9_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_10_12_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_13_15_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_16_18_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_19_21_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_22_24_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_25_27_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_28_30_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_31_33_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_34_36_features.set_index('timestamp_ccentral'),
                                        dschrg_pnwa_37_39_features.set_index('timestamp_ccentral')
                                        ],
                                          axis=1)

In [20]:
feature_checks(dschrg_pnwa_final_features)
dschrg_pnwa_final_features.head()

columns Index(['dschrg_pnwa_mean_1_3', 'dschrg_pnwa_mean_4_6', 'dschrg_pnwa_mean_7_9',
       'dschrg_pnwa_mean_10_12', 'dschrg_pnwa_mean_13_15',
       'dschrg_pnwa_mean_16_18', 'dschrg_pnwa_mean_19_21',
       'dschrg_pnwa_mean_22_24', 'dschrg_pnwa_mean_25_27',
       'dschrg_pnwa_mean_28_30', 'dschrg_pnwa_mean_31_33',
       'dschrg_pnwa_mean_34_36', 'dschrg_pnwa_mean_37_39'],
      dtype='object')
start time 2020-12-31 09:00:00-06:00
end time 2023-12-31 07:00:00-06:00
number of rows 26280 and columns 13.000000
Count of non-null values by column


,dschrg_pnwa_mean_1_3,dschrg_pnwa_mean_4_6,dschrg_pnwa_mean_7_9,dschrg_pnwa_mean_10_12,dschrg_pnwa_mean_13_15,dschrg_pnwa_mean_16_18,dschrg_pnwa_mean_19_21,dschrg_pnwa_mean_22_24,dschrg_pnwa_mean_25_27,dschrg_pnwa_mean_28_30,dschrg_pnwa_mean_31_33,dschrg_pnwa_mean_34_36,dschrg_pnwa_mean_37_39
timestamp_ccentral,,,,,,,,,,,,,
2020-12-31 08:00:00-06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 09:00:00-06:00,732.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 10:00:00-06:00,641.450000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 11:00:00-06:00,607.366667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31 12:00:00-06:00,543.900000,732.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Export dataset as csv (uncomment if need to re-export)
dschrg_pnwa_final_features.to_csv(os.path.join(project_conf.INTERIM_DATA_DIR, 'discharge_pnwa_features.csv'), index=True)

## 2. Combine Previous Feature Sets Generated

In [22]:
from src.feature_creation import read_and_merge_csv

#All the toc and tur files
toc_files = [
    'toc_pnwa_concentration_features.csv',
    'toc_pnwa_load_features.csv',
    'toc_wmth_concentration_features.csv',
    'toc_wmth_load_features.csv'
]

tur_files = [
    'tur_pnwa_concentration_features.csv',
    'tur_pnwa_load_features.csv',
    'tur_wmth_concentration_features.csv',
    'tur_wmth_load_features.csv'
]

discharge_files = [
    'discharge_pnwa_features.csv',
    'discharge_wmth_features.csv'
]

#Combine all toc files
toc_combined_df = read_and_merge_csv(toc_files, '../data/interim/')

#Combine all tur files
tur_combined_df = read_and_merge_csv(tur_files, '../data/interim/')

#Combine all dischargefiles
discharge_combined_df = read_and_merge_csv(discharge_files, '../data/interim/')

#Combine the discharge data with toc and tur combined data
toc_merged_df = pd.merge(toc_combined_df, discharge_combined_df, on="timestamp_ccentral", how="outer")
tur_merged_df = pd.merge(tur_combined_df, discharge_combined_df, on="timestamp_ccentral", how="outer")

#Save the combined csv files
toc_merged_df.to_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_input_features.csv'), index=False)
tur_merged_df.to_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_input_features.csv'), index=False)


In [23]:

# ## Refer to files:
# ## 1. 3h_toc_input_features.csv
# ## 2. 3h_tur_input_features.csv

# # Glimpse the combined feature sets
# toc_merged_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_input_features.csv'))
# tur_merged_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_input_features.csv'))

# view_toc.head()
# # view_tur.head()


In [24]:
toc_merged_df.shape

(26831, 91)

## 3. Feature Sets with Pine Falls Discharge Data Added

In [25]:

## Coded join for TOC below
# Load the CSV files
input_toc_features_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_input_features.csv'))
discharge_df = pd.read_csv(os.path.join(project_conf.RAW_DATA_DIR,'pine_falls_disch.csv'))

# Directly convert timestamps to datetime with UTC
input_toc_features_df['timestamp_ccentral'] = pd.to_datetime(input_toc_features_df['timestamp_ccentral'], utc=True)
discharge_df['timestamp'] = pd.to_datetime(discharge_df['timestamp'], utc=True)

# Merge the dataframes on the timestamps
merged_df = pd.merge(input_toc_features_df, discharge_df, left_on='timestamp_ccentral', right_on='timestamp', how='left')

# Drop the redundant timestamp column from discharge_df
toc_merged_df = merged_df.drop(columns=['timestamp'])

# Save the merged dataframe to a CSV file
toc_merged_df.to_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_xy_full.csv'), index = False)

# Display the first few rows of the merged dataframe to ensure correctness
toc_merged_df[1:10]



,timestamp_ccentral,toc_pnwa_mean_1_3,toc_pnwa_mean_4_6,toc_pnwa_mean_7_9,toc_pnwa_mean_10_12,toc_pnwa_mean_13_15,toc_pnwa_mean_16_18,toc_pnwa_mean_19_21,toc_pnwa_mean_22_24,toc_pnwa_mean_25_27,...,dschrg_wmth_mean_13_15,dschrg_wmth_mean_16_18,dschrg_wmth_mean_19_21,dschrg_wmth_mean_22_24,dschrg_wmth_mean_25_27,dschrg_wmth_mean_28_30,dschrg_wmth_mean_31_33,dschrg_wmth_mean_34_36,dschrg_wmth_mean_37_39,discharge
1,2020-12-31 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-12-31 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-12-31 17:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-31 18:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2020-12-31 19:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,519.1
6,2020-12-31 20:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,521.3
7,2020-12-31 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,522.1
8,2020-12-31 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,522.6
9,2020-12-31 23:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,521.0


In [26]:
## Coded join for TUR below
# Load the CSV files
input_tur_features_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_input_features.csv'))
discharge_df = pd.read_csv(os.path.join(project_conf.RAW_DATA_DIR,'pine_falls_disch.csv'))

# Directly convert timestamps to datetime with UTC
input_tur_features_df['timestamp_ccentral'] = pd.to_datetime(input_tur_features_df['timestamp_ccentral'], utc=True)
discharge_df['timestamp'] = pd.to_datetime(discharge_df['timestamp'], utc=True)

# Merge the dataframes on the timestamps
merged_df = pd.merge(input_tur_features_df, discharge_df, left_on='timestamp_ccentral', right_on='timestamp', how='left')

# Drop the redundant timestamp column from discharge_df
tur_merged_df = merged_df.drop(columns=['timestamp'])

# Save the merged dataframe to a CSV file
merged_df.to_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_xy_full.csv'), index = False)

# Display the first few rows of the merged dataframe to ensure correctness
tur_merged_df[1:10]

,timestamp_ccentral,turbidity_pnwa_mean_1_3,turbidity_pnwa_mean_4_6,turbidity_pnwa_mean_7_9,turbidity_pnwa_mean_10_12,turbidity_pnwa_mean_13_15,turbidity_pnwa_mean_16_18,turbidity_pnwa_mean_19_21,turbidity_pnwa_mean_22_24,turbidity_pnwa_mean_25_27,...,dschrg_wmth_mean_13_15,dschrg_wmth_mean_16_18,dschrg_wmth_mean_19_21,dschrg_wmth_mean_22_24,dschrg_wmth_mean_25_27,dschrg_wmth_mean_28_30,dschrg_wmth_mean_31_33,dschrg_wmth_mean_34_36,dschrg_wmth_mean_37_39,discharge
1,2020-12-31 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-12-31 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-12-31 17:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-31 18:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2020-12-31 19:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,519.1
6,2020-12-31 20:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,521.3
7,2020-12-31 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,522.1
8,2020-12-31 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,522.6
9,2020-12-31 23:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,521.0


## y = pf_toc, TOC at Pine Falls data filtered

In [27]:
# Extracting toc data at pine falls
# Filter the data for site 'pf' and parameter 'toc'
filtered_data = full_dataset[(full_dataset['site'] == 'pf') & (full_dataset['parameter'] == 'toc')]

# Select only the 'timestamp' and 'value' columns
pf_toc = filtered_data[['timestamp', 'value']]

# pf_toc.to_csv(os.path.join(project_conf.RAW_DATA_DIR, 'pf_toc.csv'))

# Display the result
pf_toc.head()

,timestamp,value
105633,2021-08-24T20:00:00Z,11.2414
105636,2021-08-24T21:00:00Z,11.2514
105639,2021-08-24T22:00:00Z,11.2914
105642,2021-08-24T23:00:00Z,11.2514
105645,2021-08-25T00:00:00Z,11.2014


## y = pf_tur, TUR at Pine Falls data filtered

In [28]:
# Extracting tur data at pine falls
# Filter the data for site 'pf' and parameter 'toc'
filtered_data = full_dataset[(full_dataset['site'] == 'pf') & (full_dataset['parameter'] == 'turbidity')]

# Select only the 'timestamp' and 'value' columns
pf_tur = filtered_data[['timestamp', 'value']]

pf_tur.to_csv(os.path.join(project_conf.RAW_DATA_DIR, 'pf_tur.csv'))

# Display the result
pf_tur.head()

,timestamp,value
105634,2021-08-24T20:00:00Z,8.4422
105637,2021-08-24T21:00:00Z,8.5322
105640,2021-08-24T22:00:00Z,8.3322
105643,2021-08-24T23:00:00Z,8.5422
105646,2021-08-25T00:00:00Z,8.1822


## TOC at Pine Falls Joins the Full xy Data

In [29]:
# Load the CSV files
pf_toc_df = pd.read_csv(os.path.join(project_conf.RAW_DATA_DIR, 'pf_toc.csv'))
toc_xy_full_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_xy_full.csv'))

# Convert 'timestamp_ccentral' to datetime in toc_xy_full_df
toc_xy_full_df['timestamp_ccentral'] = pd.to_datetime(toc_xy_full_df['timestamp_ccentral'], utc=True)

# Convert 'timestamp' to datetime in pf_toc_df
pf_toc_df['timestamp'] = pd.to_datetime(pf_toc_df['timestamp'])

# Merge the dataframes on the standardized timestamp columns
merged_xy_full_df = pd.merge(toc_xy_full_df, pf_toc_df[['timestamp', 'value']], how='left', left_on='timestamp_ccentral', right_on='timestamp')

# Drop the redundant timestamp column from pf_toc_df
merged_xy_full_df = merged_xy_full_df.drop(columns=['timestamp'])

# Save the merged dataframe to a CSV file
merged_xy_full_df.to_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_xy_full.csv'), index = False)

# Display the first few rows of the merged dataframe to ensure correctness
merged_xy_full_df.head()


,timestamp_ccentral,toc_pnwa_mean_1_3,toc_pnwa_mean_4_6,toc_pnwa_mean_7_9,toc_pnwa_mean_10_12,toc_pnwa_mean_13_15,toc_pnwa_mean_16_18,toc_pnwa_mean_19_21,toc_pnwa_mean_22_24,toc_pnwa_mean_25_27,...,dschrg_wmth_mean_16_18,dschrg_wmth_mean_19_21,dschrg_wmth_mean_22_24,dschrg_wmth_mean_25_27,dschrg_wmth_mean_28_30,dschrg_wmth_mean_31_33,dschrg_wmth_mean_34_36,dschrg_wmth_mean_37_39,discharge,value
0,2020-12-31 14:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-12-31 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-12-31 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-12-31 17:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-31 18:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## TUR at Pine Falls Joins the Full xy Data

In [30]:
# Load the CSV files
pf_tur_df = pd.read_csv(os.path.join(project_conf.RAW_DATA_DIR, 'pf_tur.csv'))
tur_xy_full_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_xy_full.csv'))

# Convert 'timestamp_ccentral' and 'timestamp' to datetime in both dataframes
tur_xy_full_df['timestamp_ccentral'] = pd.to_datetime(tur_xy_full_df['timestamp_ccentral'], utc=True)
pf_tur_df['timestamp'] = pd.to_datetime(pf_tur_df['timestamp'], utc=True)

# Merge the dataframes on the standardized timestamp columns
merged_xy_full_df = pd.merge(tur_xy_full_df, pf_tur_df[['timestamp', 'value']], how='left', 
                             left_on='timestamp_ccentral', right_on='timestamp')

# Drop the redundant timestamp columns from the merged dataframe
merged_xy_full_df.drop(columns=['timestamp_x', 'timestamp_y'], inplace=True)

merged_xy_full_df.to_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_xy_full.csv'), index = False)

# Display the first few rows of the cleaned and aligned dataframe
merged_xy_full_df[1:10]


,timestamp_ccentral,turbidity_pnwa_mean_1_3,turbidity_pnwa_mean_4_6,turbidity_pnwa_mean_7_9,turbidity_pnwa_mean_10_12,turbidity_pnwa_mean_13_15,turbidity_pnwa_mean_16_18,turbidity_pnwa_mean_19_21,turbidity_pnwa_mean_22_24,turbidity_pnwa_mean_25_27,...,dschrg_wmth_mean_16_18,dschrg_wmth_mean_19_21,dschrg_wmth_mean_22_24,dschrg_wmth_mean_25_27,dschrg_wmth_mean_28_30,dschrg_wmth_mean_31_33,dschrg_wmth_mean_34_36,dschrg_wmth_mean_37_39,discharge,value
1,2020-12-31 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-12-31 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-12-31 17:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-31 18:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2020-12-31 19:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,519.1,NaN
6,2020-12-31 20:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,521.3,NaN
7,2020-12-31 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,522.1,NaN
8,2020-12-31 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,522.6,NaN
9,2020-12-31 23:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,521.0,NaN


In [31]:

# # Load the 3h_toc_xy_full.csv to inspect its structure for reference
# reference_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_xy_full.csv'))
# reference_columns = reference_df.columns.tolist()

# # Map current column names to the desired structure
# column_mapping = {
#     'timestamp_ccentral': 'timestamp_ccentral',
#     'value': 'value',
#     # Add any additional mappings if necessary
# }

# # Rename columns in the merged dataframe according to the mapping
# merged_xy_full_df.rename(columns=column_mapping, inplace=True)

# # Ensure all necessary columns are included and in the correct order
# for col in reference_columns:
#     if col not in merged_xy_full_df.columns:
#         merged_xy_full_df[col] = None  # Add missing columns with None values

# # Reorder columns to match the reference structure
# merged_xy_full_df = merged_xy_full_df[reference_columns]



